# Reflection Design Pattern — Lab 1: Chart Reflection

An agent that generates a chart, looks at its own output, critiques it, and
regenerates an improved version.

Design decisions and references are in `README.md`. The logic lives in `src/`;
this notebook just wires it together and shows the result.


In [ ]:
# Auto-reload src/ modules on every cell run (handy while iterating).
%load_ext autoreload
%autoreload 2

from dotenv import load_dotenv

from src import config
from src.rendering import load_and_prepare_data, print_html
from src.agent import build_agent

load_dotenv()  # loads OPENAI_API_KEY from .env


In [ ]:
# 1. Load the data once. The agent's tools will use this dataframe via a closure.
df = load_and_prepare_data(str(config.DATASET_PATH))
print_html(df.sample(n=5), title="Random sample of coffee sales")


In [ ]:
# 2. Build the agent (model + the two tools + system prompt).
agent = build_agent(df)


In [ ]:
# 3. Run it. We pass the same instruction the tools were configured with.
#    thread_id lets the checkpointer keep this run's memory together.
result = agent.invoke(
    {"messages": [{"role": "user", "content": config.INSTRUCTION}]},
    config={"configurable": {"thread_id": "chart-reflection-1"}},
)

# The agent's final text answer:
print(result["messages"][-1].content)


In [ ]:
# Show the run once: what each tool returned, then both charts.
from langchain_core.messages import ToolMessage
from src.rendering import print_code_html, print_html

for m in result["messages"]:
    if isinstance(m, ToolMessage):
        print_code_html(m.content, title=f"Tool output: {m.name}")

print_html(str(config.CHART_V1_PATH), title="Chart V1", is_image=True)
print_html(str(config.CHART_V2_PATH), title="Chart V2", is_image=True)


## What just happened

1. The orchestrator read the tool docstrings and called `generate_chart_v1` first.
2. That tool asked the model for pandas+matplotlib code, ran it, and saved `outputs/chart_v1.png`.
3. The orchestrator then called `reflect_and_regenerate`, which sent the V1 chart **image**
   back to the model, got a critique plus improved code, and saved `outputs/chart_v2.png`.

Both charts and the feedback are displayed inline by the tools as they run.
